# 03 - Churn Labeling

## Objective
Assign churn labels to every user based on inactivity days.

## Threshold Decision
Original framework used 25/30 day thresholds. Adjusted to 7/14 days
because the dataset's maximum inactivity is 19 days — original 
thresholds would have labeled all 1,000 users as Active, 
which provides no analytical value.

## Final Thresholds
- Active:          inactivity < 7 days
- Potential Churn: inactivity 7 to 14 days
- Churned:         inactivity ≥ 15 days

## Validation
- All 1,000 users successfully labeled
- Churned users inactivity range: 15 to 19 days
- Potential churn range: exactly 7 to 14 days
- Zero null labels

## Output
- churn_labeled.csv: 1,000 rows × 12 columns
- Active: 928 (92.8%)
- Potential Churn: 66 (6.6%)
- Churned: 6 (0.6%)

In [5]:
import pandas as pd
import numpy as np


#load the dataset
user_features = pd.read_csv(r'F:\Projects\User Retention Study\User Churn Analysis\Data\cleaned\user_features.csv')

#reconvert  the date column

user_features['first_activity'] = pd.to_datetime(user_features['first_activity'])
user_features['last_activity'] = pd.to_datetime(user_features['last_activity'])


#check
print(user_features.shape)
print(user_features.dtypes)

(1000, 10)
UserID                         int64
total_events                   int64
total_sessions                 int64
total_purchases                int64
total_spend                  float64
unique_events                  int64
first_activity        datetime64[us]
last_activity         datetime64[us]
active_days                    int64
purchase_frequency           float64
dtype: object


In [6]:
dataset_end_date = pd.Timestamp('2024-07-24')

#calculate inactive days for each user

user_features['inactivity_days'] = (
    dataset_end_date - user_features['last_activity']
).dt.days

#check
print(user_features[['UserID', 'last_activity', 'inactivity_days']].head(10))

print("\n Inactivity days stats:")
print(user_features['inactivity_days'].describe())

   UserID              last_activity  inactivity_days
0       1 2024-07-22 20:10:14.181302                1
1       2 2024-07-22 08:54:54.594265                1
2       3 2024-07-23 17:38:50.228803                0
3       4 2024-07-23 19:57:57.196352                0
4       5 2024-07-24 10:05:47.264077               -1
5       6 2024-07-23 14:58:42.224360                0
6       7 2024-07-17 15:42:50.274862                6
7       8 2024-07-22 06:07:53.771228                1
8       9 2024-07-19 17:32:15.783554                4
9      10 2024-07-23 22:50:17.279206                0

 Inactivity days stats:
count    1000.000000
mean        1.610000
std         2.784898
min        -1.000000
25%         0.000000
50%         1.000000
75%         2.000000
max        19.000000
Name: inactivity_days, dtype: float64


In [7]:
dataset_end_date = user_features['last_activity'].max()
print("Dataset end date:", dataset_end_date)

Dataset end date: 2024-07-24 10:13:04.983908


In [8]:
dataset_end_date = user_features['last_activity'].max()

user_features['inactivity_days'] = (
    dataset_end_date - user_features['last_activity']
).dt.days

print("Dataset end date:", dataset_end_date)
print("\nInactivity distribution:")
print(user_features['inactivity_days'].describe())
print("\nValue counts (top 20):")
print(user_features['inactivity_days'].value_counts().sort_index().tail(20))

Dataset end date: 2024-07-24 10:13:04.983908

Inactivity distribution:
count    1000.000000
mean        2.102000
std         2.750848
min         0.000000
25%         0.000000
50%         1.000000
75%         3.000000
max        19.000000
Name: inactivity_days, dtype: float64

Value counts (top 20):
inactivity_days
0     342
1     207
2     152
3     102
4      70
5      27
6      28
7      15
8      15
9      13
10     11
11      5
12      2
13      1
14      4
15      1
16      2
17      1
19      2
Name: count, dtype: int64


In [ ]:
# assigning churn labeling function
def assign_churn_label(days):
    if days < 7:
        return 'Active'
    elif 7 <= days <= 14:
        return 'Potential Churn'
    else:
        return 'Churned'

# Apply to every user
user_features['churn_label'] = user_features['inactivity_days'].apply(assign_churn_label)

# Check 
print("Churn Label Distribution:")
print(user_features['churn_label'].value_counts())
print("\nPercentage breakdown:")
print((user_features['churn_label'].value_counts(normalize=True) * 100).round(2))

Churn Label Distribution:
churn_label
Active             928
Potential Churn     66
Churned              6
Name: count, dtype: int64

Percentage breakdown:
churn_label
Active             92.8
Potential Churn     6.6
Churned             0.6
Name: proportion, dtype: float64


In [11]:
#
print("Total user labled:", user_features['churn_label'].notna().sum())

#check the inactivity days for churned user
print("\nChurned user detail:")
print(user_features[user_features['churn_label']== 'Churned'][['UserID', 'inactivity_days', 'last_activity']])

#check the potential churn users
print("\nPoitential churn users - inactivity range:")
potential = user_features[user_features['churn_label'] == 'Potential Churn']

print(f"Min inactivity: {potential['inactivity_days'].min()}")
print(f"Max inactivity: {potential['inactivity_days'].max()}")


Total user labled: 1000

Churned user detail:
     UserID  inactivity_days              last_activity
44       45               19 2024-07-05 04:41:25.349248
209     210               15 2024-07-09 07:30:54.721089
368     369               17 2024-07-07 00:02:40.311599
463     464               16 2024-07-07 17:08:07.576523
899     900               16 2024-07-08 08:36:15.748397
981     982               19 2024-07-04 22:32:02.202059

Poitential churn users - inactivity range:
Min inactivity: 7
Max inactivity: 14


In [12]:
user_features.to_csv(r'F:\Projects\User Retention Study\User Churn Analysis\Data\cleaned\churn_labeled.csv', index=False)

print("File saved successfully")
print(f"Final shape: {user_features.shape}")
print(f"\nFinal lable distribution:")
print(user_features['churn_label'].value_counts())

File saved successfully
Final shape: (1000, 12)

Final lable distribution:
churn_label
Active             928
Potential Churn     66
Churned              6
Name: count, dtype: int64
